In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# =========================
# 1. TÌM FILE TỰ ĐỘNG
# =========================

CANDIDATE_TRAIN = [
    Path("train.csv"),
    Path("hbaac-round2/train.csv"),
    Path("./hbaac-round2/train.csv"),
]

CANDIDATE_SAMPLE = [
    Path("sample_submission.csv"),
    Path("hbaac-round2/sample_submission.csv"),
    Path("./hbaac-round2/sample_submission.csv"),
]

train_path = next((p for p in CANDIDATE_TRAIN if p.exists()), None)
sample_path = next((p for p in CANDIDATE_SAMPLE if p.exists()), None)

assert train_path is not None, "Không tìm thấy train.csv"
assert sample_path is not None, "Không tìm thấy sample_submission.csv"

print("Train path:", train_path)
print("Sample path:", sample_path)

# =========================
# 2. ĐỌC DỮ LIỆU
# =========================

train = pd.read_csv(train_path, low_memory=False)
sample = pd.read_csv(sample_path)

print("Train shape:", train.shape)
print("Sample shape:", sample.shape)

display(train.head())
display(sample.head())

# =========================
# 3. LÀM SẠCH DỮ LIỆU
# =========================

train["Date"] = pd.to_datetime(train["Date"])
train = train.rename(columns={"Cost Amount": "CostAmount", "Unit Cost": "UnitCost"})

for col in ["SalesAmount", "CostAmount", "UnitPrice", "UnitCost"]:
    if col in train.columns:
        train[col] = (
            train[col]
            .astype(str)
            .str.replace(",", ".", regex=False)
        )
        train[col] = pd.to_numeric(train[col], errors="coerce").fillna(0)

train["Quantity"] = pd.to_numeric(train["Quantity"], errors="coerce").fillna(0)

sample["sku"] = sample["id"].str.rsplit("_", n=1).str[0]
all_skus = sorted(sample["sku"].unique())

print("Date range:", train["Date"].min(), "→", train["Date"].max())
print("Number of SKUs:", len(all_skus))
print("Negative quantity rows:", (train["Quantity"] < 0).sum())

# =========================
# 4. TÍNH PROFIT WEIGHT
# =========================

profit = (
    train.assign(profit=train["SalesAmount"] - train["CostAmount"])
    .groupby("ItemCode")["profit"]
    .sum()
    .reindex(all_skus)
    .fillna(0)
    .clip(lower=0)
)

weights = profit / profit.sum()

weight_table = (
    pd.DataFrame({
        "sku": weights.index,
        "profit": profit.values,
        "weight": weights.values
    })
    .sort_values("weight", ascending=False)
)

weight_table["cum_weight"] = weight_table["weight"].cumsum()

display(weight_table.head(20))

for k in [10, 100, 1000, 2000, 5000]:
    print(f"Top {k} SKU cumulative weight:", round(weight_table.head(k)["weight"].sum(), 4))

# =========================
# 5. TẠO DAILY TIME SERIES
# =========================

daily = (
    train.groupby(["Date", "ItemCode"], as_index=False)["Quantity"]
    .sum()
)

dates = pd.date_range(train["Date"].min(), train["Date"].max(), freq="D")

pivot = (
    daily.pivot(index="Date", columns="ItemCode", values="Quantity")
    .reindex(index=dates, columns=all_skus, fill_value=0)
    .fillna(0)
    .astype("float32")
)

print("Pivot shape:", pivot.shape)
display(pivot.iloc[:5, :5])

# =========================
# 6. BASELINE FORECAST FUNCTION
# =========================

def make_forecast_matrix(history: pd.DataFrame, forecast_dates: pd.DatetimeIndex) -> pd.DataFrame:
    cols = history.columns

    ma28 = history.tail(28).mean(axis=0)
    ma56 = history.tail(56).mean(axis=0)
    ma84 = history.tail(84).mean(axis=0)
    last28 = history.tail(28)

    # SKU lâu không bán thì giảm forecast
    arr = history.values
    pos = arr > 0
    n = len(history)

    last_pos = np.where(
        pos.any(axis=0),
        n - 1 - np.argmax(pos[::-1], axis=0),
        -10**9
    )

    days_since = pd.Series(n - 1 - last_pos, index=cols)

    decay = pd.Series(1.0, index=cols)
    decay[days_since > 60] = 0.7
    decay[days_since > 90] = 0.5
    decay[days_since > 180] = 0.25
    decay[days_since > 365] = 0.05
    decay[days_since > 730] = 0.0

    # Trung bình theo thứ trong tuần
    last84 = history.tail(84)
    weekday_means = {}

    for wd in range(7):
        subset = last84[last84.index.dayofweek == wd]
        weekday_means[wd] = subset.mean(axis=0) if len(subset) > 0 else ma84

    preds = []

    for j, d in enumerate(forecast_dates):
        pattern = last28.iloc[j % 28] if len(last28) == 28 else ma28

        base = (
            0.35 * ma28
            + 0.20 * ma56
            + 0.30 * weekday_means[d.dayofweek]
            + 0.15 * pattern
        )

        pred = (base * decay).clip(lower=0)
        preds.append(pred.values.astype("float32"))

    return pd.DataFrame(preds, index=forecast_dates, columns=cols)


def compute_weights(train_df, sku_order, cutoff=None):
    df = train_df if cutoff is None else train_df[train_df["Date"] <= pd.to_datetime(cutoff)]

    profit = (
        df.assign(profit=df["SalesAmount"] - df["CostAmount"])
        .groupby("ItemCode")["profit"]
        .sum()
        .reindex(sku_order)
        .fillna(0)
        .clip(lower=0)
    )

    return profit / profit.sum()


def wrmsse_score(history, actual, pred, weights, eps=1e-6):
    scale = (history.diff().iloc[1:] ** 2).mean(axis=0).astype("float64")
    mse = ((actual - pred) ** 2).mean(axis=0).astype("float64")

    rmsse = np.sqrt(mse / (scale + eps))
    rmsse = rmsse.replace([np.inf, -np.inf], np.nan).fillna(0)

    score = float((rmsse * weights.reindex(rmsse.index).fillna(0)).sum())
    return score, rmsse

# =========================
# 7. LOCAL VALIDATION
# =========================

last_date = pivot.index.max()
cutoff = last_date - pd.Timedelta(days=28)

valid_dates = pd.date_range(cutoff + pd.Timedelta(days=1), periods=28, freq="D")

history = pivot.loc[:cutoff]
actual = pivot.loc[valid_dates]

pred_valid = make_forecast_matrix(history, valid_dates)
local_weights = compute_weights(train, pivot.columns, cutoff=cutoff)

score, rmsse_by_sku = wrmsse_score(history, actual, pred_valid, local_weights)

print("Local validation period:", valid_dates.min().date(), "→", valid_dates.max().date())
print("Local WRMSSE baseline:", round(score, 6))

# =========================
# 8. TẠO FILE SUBMISSION
# =========================

future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=28, freq="D")
pred_future = make_forecast_matrix(pivot, future_dates)

submission = sample.drop(columns=["sku"]).copy()
sku_order = sample["sku"].values

for h in range(1, 29):
    submission[f"F{h}"] = pred_future.loc[future_dates[h - 1], sku_order].values

# Nếu BTC bắt buộc số nguyên thì mở 3 dòng dưới đây:
# for h in range(1, 29):
#     submission[f"F{h}"] = submission[f"F{h}"].round().clip(lower=0).astype(int)

out_path = train_path.parent / "submission_baseline_wma.csv"
submission.to_csv(out_path, index=False)

print("Saved submission:", out_path)
display(submission.head())
print(submission.shape)

ModuleNotFoundError: No module named 'pandas'

In [2]:
!pip install pandas numpy

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   ------ --------------------------------- 1.6/9.9 MB 8.4 MB/s eta 0:00:01
   ------------- -------------------------- 3.4/9.9 MB 7.8 MB/s eta 0:00:01
   ---------------------- ----------------- 5.5/9.9 MB 7.7 MB/s eta 0:00:01
   ---------------------------- ----------- 7.1/9.9 MB 7.8 MB/s eta 0:00:01
   ------------------------------------ --- 8.9/9.9 MB 7.8 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 7.7 MB/s  0:00:01
   ---------------------------------------- 0.0/12.5 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/12.5 MB 8.2 MB/s eta 0:00:02
   ---------- ----------------------------- 3.4/12.5 MB 8.4 MB/s eta 0:00:02
   ---------------- ----------------------- 5.2/12.5 MB 8.4 MB/s eta 0:00:01
   --------------------- ------------------ 6.8/12.5 MB 8.3 MB/s eta 0:00:01
   --------------------------- 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# =========================
# 1. TÌM FILE TỰ ĐỘNG
# =========================

CANDIDATE_TRAIN = [
    Path("train.csv"),
    Path("hbaac-round2/train.csv"),
    Path("./hbaac-round2/train.csv"),
]

CANDIDATE_SAMPLE = [
    Path("sample_submission.csv"),
    Path("hbaac-round2/sample_submission.csv"),
    Path("./hbaac-round2/sample_submission.csv"),
]

train_path = next((p for p in CANDIDATE_TRAIN if p.exists()), None)
sample_path = next((p for p in CANDIDATE_SAMPLE if p.exists()), None)

assert train_path is not None, "Không tìm thấy train.csv"
assert sample_path is not None, "Không tìm thấy sample_submission.csv"

print("Train path:", train_path)
print("Sample path:", sample_path)

# =========================
# 2. ĐỌC DỮ LIỆU
# =========================

train = pd.read_csv(train_path, low_memory=False)
sample = pd.read_csv(sample_path)

print("Train shape:", train.shape)
print("Sample shape:", sample.shape)

display(train.head())
display(sample.head())

# =========================
# 3. LÀM SẠCH DỮ LIỆU
# =========================

train["Date"] = pd.to_datetime(train["Date"])
train = train.rename(columns={"Cost Amount": "CostAmount", "Unit Cost": "UnitCost"})

for col in ["SalesAmount", "CostAmount", "UnitPrice", "UnitCost"]:
    if col in train.columns:
        train[col] = (
            train[col]
            .astype(str)
            .str.replace(",", ".", regex=False)
        )
        train[col] = pd.to_numeric(train[col], errors="coerce").fillna(0)

train["Quantity"] = pd.to_numeric(train["Quantity"], errors="coerce").fillna(0)

sample["sku"] = sample["id"].str.rsplit("_", n=1).str[0]
all_skus = sorted(sample["sku"].unique())

print("Date range:", train["Date"].min(), "→", train["Date"].max())
print("Number of SKUs:", len(all_skus))
print("Negative quantity rows:", (train["Quantity"] < 0).sum())

# =========================
# 4. TÍNH PROFIT WEIGHT
# =========================

profit = (
    train.assign(profit=train["SalesAmount"] - train["CostAmount"])
    .groupby("ItemCode")["profit"]
    .sum()
    .reindex(all_skus)
    .fillna(0)
    .clip(lower=0)
)

weights = profit / profit.sum()

weight_table = (
    pd.DataFrame({
        "sku": weights.index,
        "profit": profit.values,
        "weight": weights.values
    })
    .sort_values("weight", ascending=False)
)

weight_table["cum_weight"] = weight_table["weight"].cumsum()

display(weight_table.head(20))

for k in [10, 100, 1000, 2000, 5000]:
    print(f"Top {k} SKU cumulative weight:", round(weight_table.head(k)["weight"].sum(), 4))

# =========================
# 5. TẠO DAILY TIME SERIES
# =========================

daily = (
    train.groupby(["Date", "ItemCode"], as_index=False)["Quantity"]
    .sum()
)

dates = pd.date_range(train["Date"].min(), train["Date"].max(), freq="D")

pivot = (
    daily.pivot(index="Date", columns="ItemCode", values="Quantity")
    .reindex(index=dates, columns=all_skus, fill_value=0)
    .fillna(0)
    .astype("float32")
)

print("Pivot shape:", pivot.shape)
display(pivot.iloc[:5, :5])

# =========================
# 6. BASELINE FORECAST FUNCTION
# =========================

def make_forecast_matrix(history: pd.DataFrame, forecast_dates: pd.DatetimeIndex) -> pd.DataFrame:
    cols = history.columns

    ma28 = history.tail(28).mean(axis=0)
    ma56 = history.tail(56).mean(axis=0)
    ma84 = history.tail(84).mean(axis=0)
    last28 = history.tail(28)

    # SKU lâu không bán thì giảm forecast
    arr = history.values
    pos = arr > 0
    n = len(history)

    last_pos = np.where(
        pos.any(axis=0),
        n - 1 - np.argmax(pos[::-1], axis=0),
        -10**9
    )

    days_since = pd.Series(n - 1 - last_pos, index=cols)

    decay = pd.Series(1.0, index=cols)
    decay[days_since > 60] = 0.7
    decay[days_since > 90] = 0.5
    decay[days_since > 180] = 0.25
    decay[days_since > 365] = 0.05
    decay[days_since > 730] = 0.0

    # Trung bình theo thứ trong tuần
    last84 = history.tail(84)
    weekday_means = {}

    for wd in range(7):
        subset = last84[last84.index.dayofweek == wd]
        weekday_means[wd] = subset.mean(axis=0) if len(subset) > 0 else ma84

    preds = []

    for j, d in enumerate(forecast_dates):
        pattern = last28.iloc[j % 28] if len(last28) == 28 else ma28

        base = (
            0.35 * ma28
            + 0.20 * ma56
            + 0.30 * weekday_means[d.dayofweek]
            + 0.15 * pattern
        )

        pred = (base * decay).clip(lower=0)
        preds.append(pred.values.astype("float32"))

    return pd.DataFrame(preds, index=forecast_dates, columns=cols)


def compute_weights(train_df, sku_order, cutoff=None):
    df = train_df if cutoff is None else train_df[train_df["Date"] <= pd.to_datetime(cutoff)]

    profit = (
        df.assign(profit=df["SalesAmount"] - df["CostAmount"])
        .groupby("ItemCode")["profit"]
        .sum()
        .reindex(sku_order)
        .fillna(0)
        .clip(lower=0)
    )

    return profit / profit.sum()


def wrmsse_score(history, actual, pred, weights, eps=1e-6):
    scale = (history.diff().iloc[1:] ** 2).mean(axis=0).astype("float64")
    mse = ((actual - pred) ** 2).mean(axis=0).astype("float64")

    rmsse = np.sqrt(mse / (scale + eps))
    rmsse = rmsse.replace([np.inf, -np.inf], np.nan).fillna(0)

    score = float((rmsse * weights.reindex(rmsse.index).fillna(0)).sum())
    return score, rmsse

# =========================
# 7. LOCAL VALIDATION
# =========================

last_date = pivot.index.max()
cutoff = last_date - pd.Timedelta(days=28)

valid_dates = pd.date_range(cutoff + pd.Timedelta(days=1), periods=28, freq="D")

history = pivot.loc[:cutoff]
actual = pivot.loc[valid_dates]

pred_valid = make_forecast_matrix(history, valid_dates)
local_weights = compute_weights(train, pivot.columns, cutoff=cutoff)

score, rmsse_by_sku = wrmsse_score(history, actual, pred_valid, local_weights)

print("Local validation period:", valid_dates.min().date(), "→", valid_dates.max().date())
print("Local WRMSSE baseline:", round(score, 6))

# =========================
# 8. TẠO FILE SUBMISSION
# =========================

future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=28, freq="D")
pred_future = make_forecast_matrix(pivot, future_dates)

submission = sample.drop(columns=["sku"]).copy()
sku_order = sample["sku"].values

for h in range(1, 29):
    submission[f"F{h}"] = pred_future.loc[future_dates[h - 1], sku_order].values

# Nếu BTC bắt buộc số nguyên thì mở 3 dòng dưới đây:
# for h in range(1, 29):
#     submission[f"F{h}"] = submission[f"F{h}"].round().clip(lower=0).astype(int)

out_path = train_path.parent / "submission_baseline_wma.csv"
submission.to_csv(out_path, index=False)

print("Saved submission:", out_path)
display(submission.head())
print(submission.shape)

Train path: hbaac-round2\train.csv
Sample path: hbaac-round2\sample_submission.csv
Train shape: (711980, 8)
Sample shape: (31944, 29)


,Date,Stt,ItemCode,Quantity,UnitPrice,SalesAmount,Unit Cost,Cost Amount
0,2020-11-17,2000004,SKU-08063,12,242700,2184300,"123559,1",1482709
1,2020-11-17,2000003,SKU-09458,600,"131818,1818",79090909,110000,66000000
2,2020-11-18,2000007,SKU-08062,6,230000,940909,101000,606000
3,2020-11-18,2000006,SKU-09458,240,270000,44181818,110000,26400000
4,2020-11-18,2000005,SKU-09458,240,270000,44181818,110000,26400000


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,SKU-00002_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,SKU-00003_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,SKU-00004_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,SKU-00005_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Date range: 2020-11-17 00:00:00 → 2025-09-05 00:00:00
Number of SKUs: 15972
Negative quantity rows: 37434


,sku,profit,weight,cum_weight
2,SKU-00003,1.670068e+10,0.096986,0.096986
1,SKU-00002,8.012686e+09,0.046532,0.143518
9197,SKU-09458,2.641121e+09,0.015338,0.158856
4,SKU-00005,2.243340e+09,0.013028,0.171883
8354,SKU-08589,1.285705e+09,0.007466,0.179350
12229,SKU-12534,1.215804e+09,0.007061,0.186410
9492,SKU-09760,1.195313e+09,0.006942,0.193352
12230,SKU-12537,1.106800e+09,0.006428,0.199780
315,SKU-00324,9.783747e+08,0.005682,0.205461
13993,SKU-14323,9.371040e+08,0.005442,0.210903


Top 10 SKU cumulative weight: 0.2109
Top 100 SKU cumulative weight: 0.3982
Top 1000 SKU cumulative weight: 0.7473
Top 2000 SKU cumulative weight: 0.8518
Top 5000 SKU cumulative weight: 0.9575
Pivot shape: (1754, 15972)


ItemCode,SKU-00001,SKU-00002,SKU-00003,SKU-00004,SKU-00005
2020-11-17,0.0,0.0,0.0,0.0,0.0
2020-11-18,0.0,0.0,0.0,0.0,0.0
2020-11-19,0.0,0.0,0.0,0.0,0.0
2020-11-20,0.0,0.0,0.0,0.0,0.0
2020-11-21,0.0,0.0,0.0,0.0,0.0


Local validation period: 2025-08-09 → 2025-09-05
Local WRMSSE baseline: 0.56074
Saved submission: hbaac-round2\submission_baseline_wma.csv


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0.135714,0.135714,0.160714,0.260714,0.535714,0.210714,0.135714,0.135714,0.135714,...,0.235714,0.360714,0.135714,0.135714,0.135714,0.160714,0.260714,0.235714,0.210714,0.135714
1,SKU-00002_validation,5.407143,2.382143,6.232143,5.232143,5.157143,6.107143,5.957143,5.407143,2.382143,...,5.157143,6.707143,5.507143,4.957143,2.382143,4.582143,4.332143,5.907143,5.807143,5.507143
2,SKU-00003_validation,12.067857,6.692857,12.042857,12.667857,12.892858,13.417858,11.967857,12.217857,6.692857,...,13.492857,17.017859,12.867857,14.467857,6.692857,9.792857,9.667857,12.742857,14.467857,10.467857
3,SKU-00004_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,SKU-00005_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


(31944, 29)


In [4]:
# =========================
# 9. PHÂN TÍCH SKU NÀO ĐANG KÉO ĐIỂM XẤU NHẤT
# =========================

diagnostic = pd.DataFrame({
    "sku": rmsse_by_sku.index,
    "rmsse": rmsse_by_sku.values,
    "weight": local_weights.reindex(rmsse_by_sku.index).fillna(0).values
})

diagnostic["weighted_error"] = diagnostic["rmsse"] * diagnostic["weight"]

# Thêm thông tin bán hàng
active_days = (history > 0).sum(axis=0)
total_qty = history.sum(axis=0)
last_7_qty = history.tail(7).sum(axis=0)
last_28_qty = history.tail(28).sum(axis=0)
last_56_qty = history.tail(56).sum(axis=0)
last_84_qty = history.tail(84).sum(axis=0)

diagnostic["active_days"] = active_days.reindex(diagnostic["sku"]).values
diagnostic["total_qty"] = total_qty.reindex(diagnostic["sku"]).values
diagnostic["last_7_qty"] = last_7_qty.reindex(diagnostic["sku"]).values
diagnostic["last_28_qty"] = last_28_qty.reindex(diagnostic["sku"]).values
diagnostic["last_56_qty"] = last_56_qty.reindex(diagnostic["sku"]).values
diagnostic["last_84_qty"] = last_84_qty.reindex(diagnostic["sku"]).values

diagnostic = diagnostic.sort_values("weighted_error", ascending=False)

print("Top 20 SKU đang kéo điểm xấu nhất:")
display(diagnostic.head(20))

print("Weighted error top 20:", diagnostic.head(20)["weighted_error"].sum())
print("Weighted error top 50:", diagnostic.head(50)["weighted_error"].sum())
print("Weighted error top 100:", diagnostic.head(100)["weighted_error"].sum())
print("Total WRMSSE:", diagnostic["weighted_error"].sum())

diagnostic.to_csv(train_path.parent / "baseline_diagnostic.csv", index=False)
print("Saved:", train_path.parent / "baseline_diagnostic.csv")

Top 20 SKU đang kéo điểm xấu nhất:


,sku,rmsse,weight,weighted_error,active_days,total_qty,last_7_qty,last_28_qty,last_56_qty,last_84_qty
2,SKU-00003,1.470983,0.095366,0.140282,1042,10579.0,82.0,272.0,566.0,831.0
1,SKU-00002,1.155909,0.046682,0.053961,875,5796.0,59.0,226.0,466.0,648.0
15241,SKU-15599,3.377639,0.002700,0.009121,865,15549.0,97.0,231.0,361.0,555.0
13993,SKU-14323,0.974340,0.005412,0.005273,816,17306.0,61.0,715.0,1063.0,1437.0
14419,SKU-14763,2.675549,0.001441,0.003855,420,4602.0,0.0,47.0,279.0,403.0
13990,SKU-14320,0.730521,0.005130,0.003748,873,15075.0,65.0,522.0,721.0,1226.0
3248,SKU-03351,2.373414,0.001476,0.003503,137,7515.0,0.0,0.0,5.0,5.0
1042,SKU-01074,3.463489,0.000845,0.002925,858,4248.0,0.0,0.0,0.0,5.0
13973,SKU-14303,1.310921,0.002104,0.002758,570,8387.0,40.0,121.0,237.0,609.0
5801,SKU-05957,2.375162,0.001044,0.002481,69,79.0,0.0,0.0,0.0,3.0


Weighted error top 20: 0.2463678345269675
Weighted error top 50: 0.284654276945597
Weighted error top 100: 0.32688751499111185
Total WRMSSE: 0.5607396636532611
Saved: hbaac-round2\baseline_diagnostic.csv


In [6]:
# =========================
# 10. KIỂM TRA BASELINE ĐANG DỰ BÁO THIẾU HAY THỪA
# =========================

error_detail = diagnostic.copy()

actual_sum = actual.sum(axis=0)
pred_sum = pred_valid.sum(axis=0)

error_detail["actual_valid_28"] = actual_sum.reindex(error_detail["sku"]).values
error_detail["pred_valid_28"] = pred_sum.reindex(error_detail["sku"]).values
error_detail["pred_minus_actual"] = error_detail["pred_valid_28"] - error_detail["actual_valid_28"]

error_detail["actual_over_pred_ratio"] = (
    error_detail["actual_valid_28"] / error_detail["pred_valid_28"].replace(0, np.nan)
)

# Trend gần đây trước validation
last_28_before_valid = history.tail(28).sum(axis=0)
prev_28_before_valid = history.iloc[-56:-28].sum(axis=0)

error_detail["last_28_before_valid"] = last_28_before_valid.reindex(error_detail["sku"]).values
error_detail["prev_28_before_valid"] = prev_28_before_valid.reindex(error_detail["sku"]).values

error_detail["recent_trend_ratio"] = (
    error_detail["last_28_before_valid"] /
    error_detail["prev_28_before_valid"].replace(0, np.nan)
)

cols = [
    "sku",
    "rmsse",
    "weight",
    "weighted_error",
    "actual_valid_28",
    "pred_valid_28",
    "pred_minus_actual",
    "actual_over_pred_ratio",
    "last_28_before_valid",
    "prev_28_before_valid",
    "recent_trend_ratio",
    "active_days"
]

display(error_detail[cols].head(30))

error_detail[cols].head(100).to_csv(train_path.parent / "top_error_detail.csv", index=False)
print("Saved:", train_path.parent / "top_error_detail.csv")

,sku,rmsse,weight,weighted_error,actual_valid_28,pred_valid_28,pred_minus_actual,actual_over_pred_ratio,last_28_before_valid,prev_28_before_valid,recent_trend_ratio,active_days
2,SKU-00003,1.470983,0.095366,0.140282,356.0,275.700012,-80.299988,1.291259,272.0,294.0,0.925170,1042
1,SKU-00002,1.155909,0.046682,0.053961,98.0,224.400009,126.400009,0.436720,226.0,240.0,0.941667,875
15241,SKU-15599,3.377639,0.002700,0.009121,1236.0,207.100006,-1028.900024,5.968131,231.0,130.0,1.776923,865
13993,SKU-14323,0.974340,0.005412,0.005273,612.0,607.500061,-4.499939,1.007407,715.0,348.0,2.054598,816
14419,SKU-14763,2.675549,0.001441,0.003855,298.0,91.700005,-206.299988,3.249727,47.0,232.0,0.202586,420
13990,SKU-14320,0.730521,0.005130,0.003748,431.0,455.699982,24.699982,0.945798,522.0,199.0,2.623116,873
3248,SKU-03351,2.373414,0.001476,0.003503,712.0,1.000000,-711.000000,712.000000,0.0,5.0,0.000000,137
1042,SKU-01074,3.463489,0.000845,0.002925,123.0,0.350000,-122.650002,351.428528,0.0,0.0,NaN,858
13973,SKU-14303,1.310921,0.002104,0.002758,394.0,145.100006,-248.899994,2.715369,121.0,116.0,1.043103,570
5801,SKU-05957,2.375162,0.001044,0.002481,7.0,0.210000,-6.790000,33.333328,0.0,0.0,NaN,69


Saved: hbaac-round2\top_error_detail.csv


In [8]:
%pip install lightgbm -q

Note: you may need to restart the kernel to use updated packages.


In [9]:
# =========================
# 12. LIGHTGBM MODEL CHO TOP-PROFIT SKU
# =========================

# Nếu máy báo chưa có lightgbm thì mở dòng dưới:
# %pip install lightgbm -q

import lightgbm as lgb
import gc

TOP_N = 3000
LOOKBACK_DAYS = 540

top_skus = weight_table.head(TOP_N)["sku"].tolist()
top_skus = [s for s in top_skus if s in pivot.columns]

print("Number of top SKUs used for LightGBM:", len(top_skus))


def make_static_features(history_df, weights_series):
    h = history_df.clip(lower=0)

    static = pd.DataFrame(index=h.columns)
    static["sku"] = h.columns
    static["weight"] = weights_series.reindex(h.columns).fillna(0).values
    static["active_days"] = (h > 0).sum(axis=0).values
    static["total_qty"] = h.sum(axis=0).values
    static["avg_qty"] = h.mean(axis=0).values
    static["avg_qty_active"] = (
        h.sum(axis=0) / (h > 0).sum(axis=0).replace(0, np.nan)
    ).fillna(0).values
    static["active_rate"] = ((h > 0).sum(axis=0) / len(h)).values
    static["last_28_sum"] = h.tail(28).sum(axis=0).values
    static["last_56_sum"] = h.tail(56).sum(axis=0).values
    static["last_84_sum"] = h.tail(84).sum(axis=0).values

    return static


def make_training_data(full_history, target_dates, sku_list, weights_series):
    ymat = full_history[sku_list].clip(lower=0).astype("float32")

    shifted = ymat.shift(1)

    rolling_7 = shifted.rolling(7, min_periods=1).mean()
    rolling_14 = shifted.rolling(14, min_periods=1).mean()
    rolling_28 = shifted.rolling(28, min_periods=1).mean()
    rolling_56 = shifted.rolling(56, min_periods=1).mean()
    rolling_84 = shifted.rolling(84, min_periods=1).mean()

    nz_28 = shifted.gt(0).rolling(28, min_periods=1).sum()
    nz_56 = shifted.gt(0).rolling(56, min_periods=1).sum()

    static = make_static_features(full_history[sku_list].loc[:max(target_dates)], weights_series)
    static = static.loc[sku_list]

    sku_code = np.arange(len(sku_list), dtype="int32")

    X_parts = []
    y_parts = []
    w_parts = []

    sku_weight = static["weight"].values.astype("float32")
    sku_weight_train = sku_weight / (sku_weight.mean() + 1e-9)
    sku_weight_train = np.clip(sku_weight_train, 0.1, 50)

    for d in target_dates:
        block = pd.DataFrame({
            "sku_code": sku_code,
            "dayofweek": d.dayofweek,
            "day": d.day,
            "month": d.month,
            "weekofyear": int(d.isocalendar().week),

            "lag_1": shifted.loc[d].values,
            "lag_7": ymat.shift(7).loc[d].values,
            "lag_14": ymat.shift(14).loc[d].values,
            "lag_28": ymat.shift(28).loc[d].values,
            "lag_56": ymat.shift(56).loc[d].values,

            "ma_7": rolling_7.loc[d].values,
            "ma_14": rolling_14.loc[d].values,
            "ma_28": rolling_28.loc[d].values,
            "ma_56": rolling_56.loc[d].values,
            "ma_84": rolling_84.loc[d].values,

            "nz_28": nz_28.loc[d].values,
            "nz_56": nz_56.loc[d].values,

            "sku_weight": static["weight"].values,
            "active_rate": static["active_rate"].values,
            "avg_qty": static["avg_qty"].values,
            "avg_qty_active": static["avg_qty_active"].values,
            "last_28_sum": static["last_28_sum"].values,
            "last_56_sum": static["last_56_sum"].values,
            "last_84_sum": static["last_84_sum"].values,
        })

        target = ymat.loc[d].values

        X_parts.append(block)
        y_parts.append(np.log1p(np.clip(target, 0, None)))
        w_parts.append(sku_weight_train)

    X = pd.concat(X_parts, ignore_index=True).fillna(0)
    y = np.concatenate(y_parts)
    w = np.concatenate(w_parts)

    return X, y, w


def make_one_step_features(current_history, forecast_date, sku_list, static):
    h = current_history[sku_list].clip(lower=0).astype("float32")

    def lag_value(lag):
        if len(h) >= lag:
            return h.iloc[-lag].values
        return np.zeros(len(sku_list), dtype="float32")

    block = pd.DataFrame({
        "sku_code": np.arange(len(sku_list), dtype="int32"),
        "dayofweek": forecast_date.dayofweek,
        "day": forecast_date.day,
        "month": forecast_date.month,
        "weekofyear": int(forecast_date.isocalendar().week),

        "lag_1": lag_value(1),
        "lag_7": lag_value(7),
        "lag_14": lag_value(14),
        "lag_28": lag_value(28),
        "lag_56": lag_value(56),

        "ma_7": h.tail(7).mean(axis=0).values,
        "ma_14": h.tail(14).mean(axis=0).values,
        "ma_28": h.tail(28).mean(axis=0).values,
        "ma_56": h.tail(56).mean(axis=0).values,
        "ma_84": h.tail(84).mean(axis=0).values,

        "nz_28": h.tail(28).gt(0).sum(axis=0).values,
        "nz_56": h.tail(56).gt(0).sum(axis=0).values,

        "sku_weight": static["weight"].values,
        "active_rate": static["active_rate"].values,
        "avg_qty": static["avg_qty"].values,
        "avg_qty_active": static["avg_qty_active"].values,
        "last_28_sum": static["last_28_sum"].values,
        "last_56_sum": static["last_56_sum"].values,
        "last_84_sum": static["last_84_sum"].values,
    })

    return block.fillna(0)


def recursive_predict_lgbm(model, history_df, forecast_dates, sku_list, weights_series):
    current_history = history_df[sku_list].copy().clip(lower=0)
    static = make_static_features(current_history, weights_series).loc[sku_list]

    preds = []

    for d in forecast_dates:
        X_step = make_one_step_features(current_history, d, sku_list, static)
        pred = np.expm1(model.predict(X_step))
        pred = np.clip(pred, 0, None).astype("float32")

        preds.append(pred)
        current_history.loc[d] = pred

    return pd.DataFrame(preds, index=forecast_dates, columns=sku_list)


# =========================
# 12.1 LOCAL VALIDATION CHO LIGHTGBM
# =========================

train_end = cutoff
train_start = train_end - pd.Timedelta(days=LOOKBACK_DAYS)

lgbm_train_dates = pd.date_range(train_start, train_end, freq="D")
lgbm_train_dates = [d for d in lgbm_train_dates if d in pivot.index]

print("Training dates:", min(lgbm_train_dates).date(), "→", max(lgbm_train_dates).date())
print("Training rows approx:", len(lgbm_train_dates) * len(top_skus))

X_train, y_train, w_train = make_training_data(
    full_history=pivot.loc[:train_end],
    target_dates=lgbm_train_dates,
    sku_list=top_skus,
    weights_series=local_weights
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.04,
    "num_leaves": 63,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "min_data_in_leaf": 80,
    "lambda_l2": 1.0,
    "verbosity": -1,
    "seed": 42,
}

dtrain = lgb.Dataset(
    X_train,
    label=y_train,
    weight=w_train,
    categorical_feature=["sku_code"],
    free_raw_data=False
)

model_lgbm = lgb.train(
    params,
    dtrain,
    num_boost_round=450,
    callbacks=[lgb.log_evaluation(100)]
)

del X_train, y_train, w_train, dtrain
gc.collect()

# Predict validation period
pred_valid_lgbm_top = recursive_predict_lgbm(
    model=model_lgbm,
    history_df=history,
    forecast_dates=valid_dates,
    sku_list=top_skus,
    weights_series=local_weights
)

# Blend thử nhiều alpha
blend_results = []

for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    # alpha = tỷ trọng LightGBM
    pred_blend = pred_valid.copy()

    pred_blend[top_skus] = (
        alpha * pred_valid_lgbm_top[top_skus]
        + (1 - alpha) * pred_valid[top_skus]
    )

    score_blend, _ = wrmsse_score(history, actual, pred_blend, local_weights)

    blend_results.append({
        "alpha_lgbm": alpha,
        "alpha_baseline": 1 - alpha,
        "local_wrmsse": score_blend
    })

blend_results = pd.DataFrame(blend_results).sort_values("local_wrmsse")
display(blend_results)

best_alpha = float(blend_results.iloc[0]["alpha_lgbm"])
best_local = float(blend_results.iloc[0]["local_wrmsse"])

print("Best alpha_lgbm:", best_alpha)
print("Best local WRMSSE:", best_local)

Number of top SKUs used for LightGBM: 3000
Training dates: 2024-02-15 → 2025-08-08
Training rows approx: 1623000
X_train: (1623000, 24)
y_train: (1623000,)


,alpha_lgbm,alpha_baseline,local_wrmsse
1,0.25,0.75,0.558173
2,0.50,0.50,0.560529
0,0.00,1.00,0.560740
3,0.75,0.25,0.567952
4,1.00,0.00,0.581184


Best alpha_lgbm: 0.25
Best local WRMSSE: 0.5581732421008081


In [10]:
# =========================
# 13. TRAIN LIGHTGBM TRÊN FULL TRAIN VÀ TẠO SUBMISSION V3
# =========================

ALPHA_LGBM = 0.25
TOP_N = 3000
LOOKBACK_DAYS = 540

top_skus_full = weight_table.head(TOP_N)["sku"].tolist()
top_skus_full = [s for s in top_skus_full if s in pivot.columns]

print("Top SKUs for final model:", len(top_skus_full))

# Train trên toàn bộ dữ liệu gần nhất
full_train_end = last_date
full_train_start = full_train_end - pd.Timedelta(days=LOOKBACK_DAYS)

full_train_dates = pd.date_range(full_train_start, full_train_end, freq="D")
full_train_dates = [d for d in full_train_dates if d in pivot.index]

print("Full training dates:", min(full_train_dates).date(), "→", max(full_train_dates).date())
print("Training rows approx:", len(full_train_dates) * len(top_skus_full))

global_weights = compute_weights(train, pivot.columns, cutoff=None)

X_full, y_full, w_full = make_training_data(
    full_history=pivot.loc[:full_train_end],
    target_dates=full_train_dates,
    sku_list=top_skus_full,
    weights_series=global_weights
)

print("X_full:", X_full.shape)
print("y_full:", y_full.shape)

dtrain_full = lgb.Dataset(
    X_full,
    label=y_full,
    weight=w_full,
    categorical_feature=["sku_code"],
    free_raw_data=False
)

model_lgbm_full = lgb.train(
    params,
    dtrain_full,
    num_boost_round=450,
    callbacks=[lgb.log_evaluation(100)]
)

del X_full, y_full, w_full, dtrain_full
gc.collect()

# =========================
# 13.1 DỰ BÁO 56 NGÀY
# validation: ngày +1 → +28
# evaluation: ngày +29 → +56
# =========================

future_56_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=56, freq="D")

# Baseline 56 ngày
pred_baseline_56 = make_forecast_matrix(pivot, future_56_dates)

# LightGBM 56 ngày cho top SKU
pred_lgbm_56_top = recursive_predict_lgbm(
    model=model_lgbm_full,
    history_df=pivot,
    forecast_dates=future_56_dates,
    sku_list=top_skus_full,
    weights_series=global_weights
)

# Blend
pred_final_56 = pred_baseline_56.copy()

pred_final_56[top_skus_full] = (
    ALPHA_LGBM * pred_lgbm_56_top[top_skus_full]
    + (1 - ALPHA_LGBM) * pred_baseline_56[top_skus_full]
)

pred_final_56 = pred_final_56.clip(lower=0)

# =========================
# 13.2 BUILD SUBMISSION FILE
# =========================

submission_v3 = sample.drop(columns=["sku"], errors="ignore").copy()

id_parts = submission_v3["id"].str.rsplit("_", n=1, expand=True)
submission_v3["sku"] = id_parts[0]
submission_v3["part"] = id_parts[1]

mask_val = submission_v3["part"] == "validation"
mask_eval = submission_v3["part"] == "evaluation"

for h in range(1, 29):
    validation_date = future_56_dates[h - 1]
    evaluation_date = future_56_dates[h - 1 + 28]

    submission_v3.loc[mask_val, f"F{h}"] = pred_final_56.loc[
        validation_date,
        submission_v3.loc[mask_val, "sku"]
    ].values

    submission_v3.loc[mask_eval, f"F{h}"] = pred_final_56.loc[
        evaluation_date,
        submission_v3.loc[mask_eval, "sku"]
    ].values

submission_v3 = submission_v3.drop(columns=["sku", "part"])

out_path_v3 = train_path.parent / "submission_v3_lgbm025_baseline075_56days.csv"
submission_v3.to_csv(out_path_v3, index=False)

print("Saved:", out_path_v3)
print("Shape:", submission_v3.shape)
display(submission_v3.head())
display(submission_v3.tail())

# Kiểm tra nhanh không có NaN/âm
print("Any NaN:", submission_v3.drop(columns=["id"]).isna().any().any())
print("Min prediction:", submission_v3.drop(columns=["id"]).min().min())
print("Max prediction:", submission_v3.drop(columns=["id"]).max().max())

Top SKUs for final model: 3000
Full training dates: 2024-03-14 → 2025-09-05
Training rows approx: 1623000
X_full: (1623000, 24)
y_full: (1623000,)


TypeError: Invalid value '[ 0.12467445  4.336951   10.620515   ...  0.          0.
  0.        ]' for dtype 'int64'

In [11]:
# =========================
# 13.2 FIX: BUILD SUBMISSION FILE V3 KHÔNG LỖI DTYPE
# =========================

# Tạo submission mới hoàn toàn, không dùng lại dtype int của sample
submission_v3 = pd.DataFrame()
submission_v3["id"] = sample["id"].values

# Ép toàn bộ F1-F28 thành float ngay từ đầu
for h in range(1, 29):
    submission_v3[f"F{h}"] = 0.0

id_parts = submission_v3["id"].str.rsplit("_", n=1, expand=True)
submission_v3["sku"] = id_parts[0]
submission_v3["part"] = id_parts[1]

mask_val = submission_v3["part"] == "validation"
mask_eval = submission_v3["part"] == "evaluation"

for h in range(1, 29):
    validation_date = future_56_dates[h - 1]
    evaluation_date = future_56_dates[h - 1 + 28]

    val_skus = submission_v3.loc[mask_val, "sku"]
    eval_skus = submission_v3.loc[mask_eval, "sku"]

    submission_v3.loc[mask_val, f"F{h}"] = pred_final_56.loc[
        validation_date,
        val_skus
    ].to_numpy(dtype=float)

    submission_v3.loc[mask_eval, f"F{h}"] = pred_final_56.loc[
        evaluation_date,
        eval_skus
    ].to_numpy(dtype=float)

submission_v3 = submission_v3.drop(columns=["sku", "part"])

out_path_v3 = train_path.parent / "submission_v3_lgbm025_baseline075_56days.csv"
submission_v3.to_csv(out_path_v3, index=False)

print("Saved:", out_path_v3)
print("Shape:", submission_v3.shape)

display(submission_v3.head())
display(submission_v3.tail())

print("Any NaN:", submission_v3.drop(columns=["id"]).isna().any().any())
print("Min prediction:", submission_v3.drop(columns=["id"]).min().min())
print("Max prediction:", submission_v3.drop(columns=["id"]).max().max())
print("Dtypes:")
print(submission_v3.dtypes.head())

Saved: hbaac-round2\submission_v3_lgbm025_baseline075_56days.csv
Shape: (31944, 29)


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0.124674,0.101786,0.158859,0.240661,0.437514,0.198724,0.140285,0.141501,0.103821,...,0.232287,0.325868,0.157118,0.145221,0.101881,0.178596,0.254389,0.219034,0.208612,0.158413
1,SKU-00002_validation,4.336951,1.786607,5.510975,4.919674,5.000800,5.554920,5.377220,4.498736,1.788060,...,5.098413,6.303901,5.194675,4.240407,1.834129,4.760784,4.563737,5.308173,5.212129,4.980485
2,SKU-00003_validation,10.620515,5.019643,10.910243,11.946266,11.328972,12.581285,10.388417,10.646212,5.019643,...,12.588313,15.698893,12.579724,12.468642,5.139259,11.219330,11.900838,11.707019,12.994612,10.339200
3,SKU-00004_validation,0.000000,0.000000,0.006263,0.040264,0.036166,0.036854,0.036854,0.013330,0.000000,...,0.083751,0.085145,0.096403,0.060728,0.003176,0.131619,0.128048,0.094037,0.096379,0.105337
4,SKU-00005_validation,0.000000,0.000000,0.005905,0.035779,0.031745,0.032423,0.032423,0.029375,0.003059,...,0.067706,0.068267,0.068267,0.073554,0.003509,0.090874,0.088297,0.073976,0.084267,0.098322


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
31939,SKU-16329_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31940,SKU-16330_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31941,SKU-16331_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31942,SKU-16332_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31943,SKU-16333_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Any NaN: False
Min prediction: 0.0
Max prediction: 148.95721435546875
Dtypes:
id        str
F1    float64
F2    float64
F3    float64
F4    float64
dtype: object


In [12]:
# =========================
# KIỂM TRA SKU WEIGHT = 0
# =========================

zero_weight_skus = weight_table[weight_table["weight"] == 0]["sku"].tolist()
positive_weight_skus = weight_table[weight_table["weight"] > 0]["sku"].tolist()

print("Total SKUs:", len(weight_table))
print("Positive-weight SKUs:", len(positive_weight_skus))
print("Zero-weight SKUs:", len(zero_weight_skus))

print("Zero-weight share:", len(zero_weight_skus) / len(weight_table))
display(weight_table.tail(20))

Total SKUs: 15972
Positive-weight SKUs: 14156
Zero-weight SKUs: 1816
Zero-weight share: 0.11369897320310543


,sku,profit,weight,cum_weight
13657,SKU-13979,0.0,0.0,1.0
13656,SKU-13978,0.0,0.0,1.0
13655,SKU-13977,0.0,0.0,1.0
13652,SKU-13974,0.0,0.0,1.0
7275,SKU-07466,0.0,0.0,1.0
13668,SKU-13991,0.0,0.0,1.0
10132,SKU-10414,0.0,0.0,1.0
4755,SKU-04888,0.0,0.0,1.0
2080,SKU-02142,0.0,0.0,1.0
10244,SKU-10528,0.0,0.0,1.0


In [13]:
# =========================
# LƯU DỮ LIỆU ĐÃ LÀM SẠCH
# =========================

from pathlib import Path

processed_dir = train_path.parent / "processed"
processed_dir.mkdir(exist_ok=True)

# 1. Lưu train đã clean
train_clean_path = processed_dir / "train_clean.csv"
train.to_csv(train_clean_path, index=False)

# 2. Lưu dữ liệu daily: Date - ItemCode - Quantity
daily_path = processed_dir / "daily_sales.csv"
daily.to_csv(daily_path, index=False)

# 3. Lưu pivot dạng pickle để không bị quá nặng và giữ đúng kiểu dữ liệu
pivot_path = processed_dir / "pivot_daily_sku.pkl"
pivot.to_pickle(pivot_path)

# 4. Lưu bảng profit weight
weight_path = processed_dir / "sku_profit_weight.csv"
weight_table.to_csv(weight_path, index=False)

print("Saved files:")
print(train_clean_path)
print(daily_path)
print(pivot_path)
print(weight_path)

Saved files:
hbaac-round2\processed\train_clean.csv
hbaac-round2\processed\daily_sales.csv
hbaac-round2\processed\pivot_daily_sku.pkl
hbaac-round2\processed\sku_profit_weight.csv


In [14]:
from pathlib import Path
import os

print("Current working directory:")
print(os.getcwd())

processed_dir = (train_path.parent / "processed").resolve()

print("\nProcessed folder absolute path:")
print(processed_dir)

print("\nFiles inside processed folder:")
for f in processed_dir.glob("*"):
    print(f)

Current working directory:
d:\HBAAC

Processed folder absolute path:
D:\HBAAC\hbaac-round2\processed

Files inside processed folder:
D:\HBAAC\hbaac-round2\processed\daily_sales.csv
D:\HBAAC\hbaac-round2\processed\pivot_daily_sku.pkl
D:\HBAAC\hbaac-round2\processed\sku_profit_weight.csv
D:\HBAAC\hbaac-round2\processed\train_clean.csv


In [15]:
# =========================
# 15. TEST NHIỀU BASELINE VARIANTS ĐỂ CẢI THIỆN WRMSSE
# =========================

def make_forecast_variant(history: pd.DataFrame, forecast_dates: pd.DatetimeIndex, variant="current") -> pd.DataFrame:
    # Clip history để tránh quantity âm làm forecast méo
    h = history.clip(lower=0).copy()
    cols = h.columns

    ma7 = h.tail(7).mean(axis=0)
    ma14 = h.tail(14).mean(axis=0)
    ma28 = h.tail(28).mean(axis=0)
    ma56 = h.tail(56).mean(axis=0)
    ma84 = h.tail(84).mean(axis=0)

    last28 = h.tail(28)

    # weekday mean trên 84 ngày gần nhất
    last84 = h.tail(84)
    weekday_means = {}

    for wd in range(7):
        subset = last84[last84.index.dayofweek == wd]
        weekday_means[wd] = subset.mean(axis=0) if len(subset) > 0 else ma84

    # days since last sale
    arr = h.values
    pos = arr > 0
    n = len(h)

    last_pos = np.where(
        pos.any(axis=0),
        n - 1 - np.argmax(pos[::-1], axis=0),
        -10**9
    )

    days_since = pd.Series(n - 1 - last_pos, index=cols)

    # decay hiện tại
    decay = pd.Series(1.0, index=cols)
    decay[days_since > 60] = 0.7
    decay[days_since > 90] = 0.5
    decay[days_since > 180] = 0.25
    decay[days_since > 365] = 0.05
    decay[days_since > 730] = 0.0

    # decay mạnh hơn cho SKU lâu không bán
    strong_decay = pd.Series(1.0, index=cols)
    strong_decay[days_since > 30] = 0.8
    strong_decay[days_since > 60] = 0.55
    strong_decay[days_since > 90] = 0.35
    strong_decay[days_since > 180] = 0.10
    strong_decay[days_since > 365] = 0.0

    # trend gần đây
    last28_sum = h.tail(28).sum(axis=0)
    prev28_sum = h.iloc[-56:-28].sum(axis=0)

    trend_ratio = last28_sum / prev28_sum.replace(0, np.nan)
    trend_ratio = trend_ratio.replace([np.inf, -np.inf], np.nan).fillna(1.0)
    trend_safe = trend_ratio.clip(lower=0.75, upper=1.25)
    trend_aggressive = trend_ratio.clip(lower=0.60, upper=1.50)

    preds = []

    for j, d in enumerate(forecast_dates):
        pattern = last28.iloc[j % 28] if len(last28) == 28 else ma28
        weekday = weekday_means[d.dayofweek]

        if variant == "current":
            base = (
                0.35 * ma28
                + 0.20 * ma56
                + 0.30 * weekday
                + 0.15 * pattern
            )
            pred = base * decay

        elif variant == "recent_heavy":
            base = (
                0.25 * ma7
                + 0.35 * ma28
                + 0.20 * weekday
                + 0.20 * pattern
            )
            pred = base * decay

        elif variant == "stable":
            base = (
                0.20 * ma28
                + 0.35 * ma56
                + 0.25 * ma84
                + 0.20 * weekday
            )
            pred = base * decay

        elif variant == "weekday_heavy":
            base = (
                0.20 * ma28
                + 0.20 * ma56
                + 0.50 * weekday
                + 0.10 * pattern
            )
            pred = base * decay

        elif variant == "pattern_heavy":
            base = (
                0.20 * ma28
                + 0.20 * ma56
                + 0.20 * weekday
                + 0.40 * pattern
            )
            pred = base * decay

        elif variant == "trend_safe":
            base = (
                0.35 * ma28
                + 0.20 * ma56
                + 0.30 * weekday
                + 0.15 * pattern
            )
            pred = base * trend_safe * decay

        elif variant == "trend_aggressive":
            base = (
                0.35 * ma28
                + 0.20 * ma56
                + 0.30 * weekday
                + 0.15 * pattern
            )
            pred = base * trend_aggressive * decay

        elif variant == "strong_decay":
            base = (
                0.35 * ma28
                + 0.20 * ma56
                + 0.30 * weekday
                + 0.15 * pattern
            )
            pred = base * strong_decay

        elif variant == "ma28_only":
            pred = ma28 * decay

        elif variant == "ma56_only":
            pred = ma56 * decay

        else:
            raise ValueError("Unknown variant")

        preds.append(pred.clip(lower=0).values.astype("float32"))

    return pd.DataFrame(preds, index=forecast_dates, columns=cols)


def evaluate_variant_on_cutoff(cutoff_date, variant):
    cutoff_date = pd.to_datetime(cutoff_date)

    valid_dates_fold = pd.date_range(
        cutoff_date + pd.Timedelta(days=1),
        periods=28,
        freq="D"
    )

    history_fold = pivot.loc[:cutoff_date]
    actual_fold = pivot.loc[valid_dates_fold]

    pred_fold = make_forecast_variant(history_fold, valid_dates_fold, variant=variant)
    weights_fold = compute_weights(train, pivot.columns, cutoff=cutoff_date)

    score_fold, _ = wrmsse_score(history_fold, actual_fold, pred_fold, weights_fold)

    return score_fold


variants = [
    "current",
    "recent_heavy",
    "stable",
    "weekday_heavy",
    "pattern_heavy",
    "trend_safe",
    "trend_aggressive",
    "strong_decay",
    "ma28_only",
    "ma56_only"
]

cutoffs = [
    "2025-06-13",
    "2025-07-11",
    "2025-08-08"
]

cv_rows = []

for variant in variants:
    scores = []

    for c in cutoffs:
        score_c = evaluate_variant_on_cutoff(c, variant)
        scores.append(score_c)

        cv_rows.append({
            "variant": variant,
            "cutoff": c,
            "valid_period": f"{pd.to_datetime(c) + pd.Timedelta(days=1):%Y-%m-%d} → {pd.to_datetime(c) + pd.Timedelta(days=28):%Y-%m-%d}",
            "wrmsse": score_c
        })

cv_results = pd.DataFrame(cv_rows)

cv_summary = (
    cv_results
    .groupby("variant")["wrmsse"]
    .agg(["mean", "std", "min", "max"])
    .sort_values("mean")
    .reset_index()
)

display(cv_summary)

best_variant = cv_summary.iloc[0]["variant"]
print("Best variant by mean CV:", best_variant)

,variant,mean,std,min,max
0,stable,0.555544,0.028851,0.528769,0.586098
1,ma28_only,0.559435,0.033317,0.526246,0.592879
2,ma56_only,0.559949,0.026540,0.534611,0.587545
3,strong_decay,0.562131,0.031217,0.532002,0.594332
4,current,0.563159,0.031102,0.533128,0.595231
5,weekday_heavy,0.563865,0.030700,0.536321,0.596963
6,recent_heavy,0.567006,0.032237,0.535706,0.600104
7,trend_safe,0.569242,0.032157,0.539110,0.603100
8,trend_aggressive,0.576038,0.033327,0.544837,0.611147
9,pattern_heavy,0.605308,0.030623,0.574806,0.636051


Best variant by mean CV: stable


In [17]:
# =========================
# 16. TẠO SUBMISSION V4: STABLE BASELINE 56 DAYS
# =========================

future_56_dates = pd.date_range(
    last_date + pd.Timedelta(days=1),
    periods=56,
    freq="D"
)

# Dự báo bằng variant stable
pred_stable_56 = make_forecast_variant(
    history=pivot,
    forecast_dates=future_56_dates,
    variant="stable"
)

# Set SKU có weight = 0 về 0
zero_weight_skus = weight_table[weight_table["weight"] == 0]["sku"].tolist()
zero_weight_skus = [s for s in zero_weight_skus if s in pred_stable_56.columns]

pred_stable_56[zero_weight_skus] = 0

print("Set zero forecast for zero-weight SKUs:", len(zero_weight_skus))

# Build submission mới, tránh lỗi dtype int/float
submission_v4 = pd.DataFrame()
submission_v4["id"] = sample["id"].values

for h in range(1, 29):
    submission_v4[f"F{h}"] = 0.0

id_parts = submission_v4["id"].str.rsplit("_", n=1, expand=True)
submission_v4["sku"] = id_parts[0]
submission_v4["part"] = id_parts[1]

mask_val = submission_v4["part"] == "validation"
mask_eval = submission_v4["part"] == "evaluation"

for h in range(1, 29):
    validation_date = future_56_dates[h - 1]
    evaluation_date = future_56_dates[h - 1 + 28]

    val_skus = submission_v4.loc[mask_val, "sku"]
    eval_skus = submission_v4.loc[mask_eval, "sku"]

    submission_v4.loc[mask_val, f"F{h}"] = pred_stable_56.loc[
        validation_date,
        val_skus
    ].to_numpy(dtype=float)

    submission_v4.loc[mask_eval, f"F{h}"] = pred_stable_56.loc[
        evaluation_date,
        eval_skus
    ].to_numpy(dtype=float)

submission_v4 = submission_v4.drop(columns=["sku", "part"])

out_path_v4 = train_path.parent / "submission_v4_stable_56days.csv"
submission_v4.to_csv(out_path_v4, index=False)

print("Saved:", out_path_v4)
print("Shape:", submission_v4.shape)

display(submission_v4.head())
display(submission_v4.tail())

print("Any NaN:", submission_v4.drop(columns=["id"]).isna().any().any())
print("Min prediction:", submission_v4.drop(columns=["id"]).min().min())
print("Max prediction:", submission_v4.drop(columns=["id"]).max().max())

Set zero forecast for zero-weight SKUs: 1816
Saved: hbaac-round2\submission_v4_stable_56days.csv
Shape: (31944, 29)


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0.158333,0.158333,0.175000,0.241667,0.225000,0.208333,0.158333,0.158333,0.158333,...,0.225000,0.208333,0.158333,0.158333,0.158333,0.175000,0.241667,0.225000,0.208333,0.158333
1,SKU-00002_validation,6.020238,4.403571,5.870238,5.703571,5.953571,5.986905,6.286904,6.020238,4.403571,...,5.953571,5.986905,6.286904,6.020238,4.403571,5.870238,5.703571,5.953571,5.986905,6.286904
2,SKU-00003_validation,12.195238,9.211905,11.278571,11.195238,11.945238,12.295238,11.728571,12.195238,9.211905,...,11.945238,12.295238,11.728571,12.195238,9.211905,11.278571,11.195238,11.945238,12.295238,11.728571
3,SKU-00004_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,SKU-00005_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
31939,SKU-16329_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31940,SKU-16330_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31941,SKU-16331_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31942,SKU-16332_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31943,SKU-16333_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Any NaN: False
Min prediction: 0.0
Max prediction: 116.29940795898438


In [18]:
# =========================
# 18. DIRECT HORIZON LIGHTGBM CHO TOP-PROFIT SKU
# =========================

import lightgbm as lgb
import gc
import numpy as np
import pandas as pd

TOP_N_DIRECT = 2000
LOOKBACK_ORIGIN_DAYS = 540
ORIGIN_STEP = 7

top_skus_direct = weight_table.head(TOP_N_DIRECT)["sku"].tolist()
top_skus_direct = [s for s in top_skus_direct if s in pivot.columns]

print("Direct model top SKUs:", len(top_skus_direct))


def _safe_take(mat, idx):
    idx = max(0, idx)
    return mat.iloc[idx].values


def make_direct_feature_block(ymat, origin_date, forecast_dates, sku_list, weights_series, make_target=True):
    """
    ymat: pivot[sku_list], clipped >= 0
    origin_date: ngày cuối cùng có dữ liệu quan sát
    forecast_dates: 28 ngày cần dự báo
    """
    idx = ymat.index.get_loc(origin_date)
    n_sku = len(sku_list)

    # Origin-based lag features
    lag_1 = _safe_take(ymat, idx)
    lag_7 = _safe_take(ymat, idx - 6)
    lag_14 = _safe_take(ymat, idx - 13)
    lag_28 = _safe_take(ymat, idx - 27)
    lag_56 = _safe_take(ymat, idx - 55)

    hist_to_origin = ymat.iloc[:idx + 1]

    ma7 = hist_to_origin.tail(7).mean(axis=0).values
    ma14 = hist_to_origin.tail(14).mean(axis=0).values
    ma28 = hist_to_origin.tail(28).mean(axis=0).values
    ma56 = hist_to_origin.tail(56).mean(axis=0).values
    ma84 = hist_to_origin.tail(84).mean(axis=0).values

    nz28 = hist_to_origin.tail(28).gt(0).sum(axis=0).values
    nz56 = hist_to_origin.tail(56).gt(0).sum(axis=0).values

    last28_sum = hist_to_origin.tail(28).sum(axis=0).values
    prev28_sum = hist_to_origin.iloc[-56:-28].sum(axis=0).values if len(hist_to_origin) >= 56 else np.zeros(n_sku)
    trend_ratio = np.divide(
        last28_sum,
        np.where(prev28_sum == 0, np.nan, prev28_sum)
    )
    trend_ratio = np.nan_to_num(trend_ratio, nan=1.0, posinf=1.0, neginf=1.0)
    trend_ratio = np.clip(trend_ratio, 0.5, 1.8)

    total_qty = hist_to_origin.sum(axis=0).values
    active_days = hist_to_origin.gt(0).sum(axis=0).values
    active_rate = active_days / len(hist_to_origin)
    avg_qty = total_qty / len(hist_to_origin)
    avg_qty_active = np.divide(
        total_qty,
        np.where(active_days == 0, np.nan, active_days)
    )
    avg_qty_active = np.nan_to_num(avg_qty_active, nan=0.0)

    sku_weight = weights_series.reindex(sku_list).fillna(0).values

    rows = []
    targets = []

    for h, d in enumerate(forecast_dates, start=1):
        # Pattern 28 ngày trước tương ứng với horizon
        pattern_idx = idx - 28 + h
        pattern_28 = _safe_take(ymat, pattern_idx)

        # Weekday mean trong 84 ngày trước origin
        last84 = hist_to_origin.tail(84)
        subset = last84[last84.index.dayofweek == d.dayofweek]
        if len(subset) > 0:
            weekday_mean_84 = subset.mean(axis=0).values
        else:
            weekday_mean_84 = ma84

        block = pd.DataFrame({
            "sku_code": np.arange(n_sku, dtype=np.int32),
            "horizon": h,
            "target_dayofweek": d.dayofweek,
            "target_day": d.day,
            "target_month": d.month,
            "target_weekofyear": int(d.isocalendar().week),

            "lag_1": lag_1,
            "lag_7": lag_7,
            "lag_14": lag_14,
            "lag_28": lag_28,
            "lag_56": lag_56,

            "ma7": ma7,
            "ma14": ma14,
            "ma28": ma28,
            "ma56": ma56,
            "ma84": ma84,

            "pattern_28": pattern_28,
            "weekday_mean_84": weekday_mean_84,

            "nz28": nz28,
            "nz56": nz56,
            "last28_sum": last28_sum,
            "prev28_sum": prev28_sum,
            "trend_ratio": trend_ratio,

            "sku_weight": sku_weight,
            "active_rate": active_rate,
            "avg_qty": avg_qty,
            "avg_qty_active": avg_qty_active,
        })

        rows.append(block)

        if make_target:
            target = ymat.loc[d].values
            targets.append(np.log1p(np.clip(target, 0, None)))

    X = pd.concat(rows, ignore_index=True).fillna(0)

    float_cols = [c for c in X.columns if c != "sku_code"]
    X[float_cols] = X[float_cols].astype("float32")

    if make_target:
        y = np.concatenate(targets).astype("float32")
        return X, y

    return X


def make_direct_training_data(history_df, cutoff_date, sku_list, weights_series):
    cutoff_date = pd.to_datetime(cutoff_date)

    ymat = history_df[sku_list].clip(lower=0).astype("float32")

    origin_start = cutoff_date - pd.Timedelta(days=LOOKBACK_ORIGIN_DAYS)
    origin_end = cutoff_date - pd.Timedelta(days=28)

    origin_dates = pd.date_range(origin_start, origin_end, freq=f"{ORIGIN_STEP}D")
    origin_dates = [d for d in origin_dates if d in ymat.index]

    print("Origin dates:", min(origin_dates).date(), "→", max(origin_dates).date())
    print("Number of origins:", len(origin_dates))
    print("Approx rows:", len(origin_dates) * 28 * len(sku_list))

    X_parts = []
    y_parts = []
    w_parts = []

    sku_weight = weights_series.reindex(sku_list).fillna(0).values
    train_weight = sku_weight / (sku_weight.mean() + 1e-9)
    train_weight = np.clip(train_weight, 0.1, 80)

    for origin in origin_dates:
        forecast_dates = pd.date_range(origin + pd.Timedelta(days=1), periods=28, freq="D")

        X_o, y_o = make_direct_feature_block(
            ymat=ymat,
            origin_date=origin,
            forecast_dates=forecast_dates,
            sku_list=sku_list,
            weights_series=weights_series,
            make_target=True
        )

        X_parts.append(X_o)
        y_parts.append(y_o)
        w_parts.append(np.tile(train_weight, 28))

    X = pd.concat(X_parts, ignore_index=True)
    y = np.concatenate(y_parts)
    w = np.concatenate(w_parts).astype("float32")

    return X, y, w


def predict_direct_lgbm(model, history_df, origin_date, forecast_dates, sku_list, weights_series):
    ymat = history_df[sku_list].clip(lower=0).astype("float32")

    X_pred = make_direct_feature_block(
        ymat=ymat,
        origin_date=pd.to_datetime(origin_date),
        forecast_dates=forecast_dates,
        sku_list=sku_list,
        weights_series=weights_series,
        make_target=False
    )

    pred = np.expm1(model.predict(X_pred))
    pred = np.clip(pred, 0, None)

    pred_matrix = pred.reshape(28, len(sku_list))

    return pd.DataFrame(
        pred_matrix,
        index=forecast_dates,
        columns=sku_list
    )


# =========================
# 18.1 LOCAL VALIDATION DIRECT LGBM
# =========================

direct_cutoff = cutoff
direct_valid_dates = valid_dates
direct_history = pivot.loc[:direct_cutoff]
direct_actual = pivot.loc[direct_valid_dates]
direct_weights = compute_weights(train, pivot.columns, cutoff=direct_cutoff)

X_direct, y_direct, w_direct = make_direct_training_data(
    history_df=direct_history,
    cutoff_date=direct_cutoff,
    sku_list=top_skus_direct,
    weights_series=direct_weights
)

print("X_direct:", X_direct.shape)
print("y_direct:", y_direct.shape)

direct_params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.035,
    "num_leaves": 63,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "min_data_in_leaf": 80,
    "lambda_l2": 2.0,
    "verbosity": -1,
    "seed": 2026,
}

dtrain_direct = lgb.Dataset(
    X_direct,
    label=y_direct,
    weight=w_direct,
    categorical_feature=["sku_code"],
    free_raw_data=False
)

model_direct = lgb.train(
    direct_params,
    dtrain_direct,
    num_boost_round=550,
    callbacks=[lgb.log_evaluation(100)]
)

del X_direct, y_direct, w_direct, dtrain_direct
gc.collect()

pred_direct_top = predict_direct_lgbm(
    model=model_direct,
    history_df=direct_history,
    origin_date=direct_cutoff,
    forecast_dates=direct_valid_dates,
    sku_list=top_skus_direct,
    weights_series=direct_weights
)

# Base vẫn là current baseline vì Public đang tốt nhất
pred_current_valid = make_forecast_variant(
    direct_history,
    direct_valid_dates,
    variant="current"
)

blend_rows = []

for alpha_direct in [0.0, 0.10, 0.20, 0.25, 0.30, 0.40, 0.50, 0.65, 0.80, 1.0]:
    pred_blend = pred_current_valid.copy()

    pred_blend[top_skus_direct] = (
        alpha_direct * pred_direct_top[top_skus_direct]
        + (1 - alpha_direct) * pred_current_valid[top_skus_direct]
    )

    score_blend, _ = wrmsse_score(
        direct_history,
        direct_actual,
        pred_blend,
        direct_weights
    )

    blend_rows.append({
        "alpha_direct_lgbm": alpha_direct,
        "alpha_current": 1 - alpha_direct,
        "local_wrmsse": score_blend
    })

direct_blend_summary = pd.DataFrame(blend_rows).sort_values("local_wrmsse")
display(direct_blend_summary)

print("Best direct blend:")
display(direct_blend_summary.head(1))

Direct model top SKUs: 2000
Origin dates: 2024-02-15 → 2025-07-10
Number of origins: 74
Approx rows: 4144000
X_direct: (4144000, 27)
y_direct: (4144000,)


,alpha_direct_lgbm,alpha_current,local_wrmsse
7,0.65,0.35,0.548230
6,0.50,0.50,0.548520
5,0.40,0.60,0.549647
8,0.80,0.20,0.549712
4,0.30,0.70,0.551490
3,0.25,0.75,0.552674
2,0.20,0.80,0.554029
9,1.00,0.00,0.554756
1,0.10,0.90,0.557244
0,0.00,1.00,0.561117


Best direct blend:


,alpha_direct_lgbm,alpha_current,local_wrmsse
7,0.65,0.35,0.54823


In [19]:
# =========================
# 19. TRAIN DIRECT LIGHTGBM FULL TRAIN + TẠO SUBMISSION V5
# =========================

ALPHA_DIRECT = 0.65
ALPHA_BASE = 0.35
TOP_N_DIRECT_FINAL = 2000

top_skus_final = weight_table.head(TOP_N_DIRECT_FINAL)["sku"].tolist()
top_skus_final = [s for s in top_skus_final if s in pivot.columns]

print("Top SKUs for final direct model:", len(top_skus_final))

global_weights = compute_weights(train, pivot.columns, cutoff=None)

# =========================
# 19.1 TRAIN DIRECT MODEL TRÊN FULL TRAIN
# =========================

X_direct_full, y_direct_full, w_direct_full = make_direct_training_data(
    history_df=pivot,
    cutoff_date=last_date,
    sku_list=top_skus_final,
    weights_series=global_weights
)

print("X_direct_full:", X_direct_full.shape)
print("y_direct_full:", y_direct_full.shape)

dtrain_direct_full = lgb.Dataset(
    X_direct_full,
    label=y_direct_full,
    weight=w_direct_full,
    categorical_feature=["sku_code"],
    free_raw_data=False
)

model_direct_full = lgb.train(
    direct_params,
    dtrain_direct_full,
    num_boost_round=550,
    callbacks=[lgb.log_evaluation(100)]
)

del X_direct_full, y_direct_full, w_direct_full, dtrain_direct_full
gc.collect()

# =========================
# 19.2 DỰ BÁO 56 NGÀY
# =========================

future_56_dates = pd.date_range(
    last_date + pd.Timedelta(days=1),
    periods=56,
    freq="D"
)

future_val_dates = future_56_dates[:28]
future_eval_dates = future_56_dates[28:]

# Base dùng lại logic baseline public tốt nhất
pred_base_56 = make_forecast_matrix(
    history=pivot,
    forecast_dates=future_56_dates
)

pred_final_56 = pred_base_56.copy()

# =========================
# 19.3 DIRECT PREDICT CHO 28 NGÀY ĐẦU
# =========================

pred_direct_val_top = predict_direct_lgbm(
    model=model_direct_full,
    history_df=pivot,
    origin_date=last_date,
    forecast_dates=future_val_dates,
    sku_list=top_skus_final,
    weights_series=global_weights
)

pred_final_56.loc[future_val_dates, top_skus_final] = (
    ALPHA_DIRECT * pred_direct_val_top[top_skus_final]
    + ALPHA_BASE * pred_base_56.loc[future_val_dates, top_skus_final]
)

# =========================
# 19.4 DIRECT PREDICT CHO 28 NGÀY TIẾP THEO
# Dùng forecast 28 ngày đầu làm history giả lập
# =========================

augmented_top_history = pivot[top_skus_final].clip(lower=0).copy()

predicted_val_top = pred_final_56.loc[future_val_dates, top_skus_final].copy()

augmented_top_history = pd.concat(
    [augmented_top_history, predicted_val_top],
    axis=0
)

pred_direct_eval_top = predict_direct_lgbm(
    model=model_direct_full,
    history_df=augmented_top_history,
    origin_date=future_val_dates[-1],
    forecast_dates=future_eval_dates,
    sku_list=top_skus_final,
    weights_series=global_weights
)

pred_final_56.loc[future_eval_dates, top_skus_final] = (
    ALPHA_DIRECT * pred_direct_eval_top[top_skus_final]
    + ALPHA_BASE * pred_base_56.loc[future_eval_dates, top_skus_final]
)

pred_final_56 = pred_final_56.clip(lower=0)

# =========================
# 19.5 SET ZERO-WEIGHT SKU = 0
# =========================

zero_weight_skus = weight_table[weight_table["weight"] == 0]["sku"].tolist()
zero_weight_skus = [s for s in zero_weight_skus if s in pred_final_56.columns]

pred_final_56[zero_weight_skus] = 0

print("Zero-weight SKUs set to 0:", len(zero_weight_skus))

# =========================
# 19.6 BUILD SUBMISSION FILE
# =========================

submission_v5 = pd.DataFrame()
submission_v5["id"] = sample["id"].values

for h in range(1, 29):
    submission_v5[f"F{h}"] = 0.0

id_parts = submission_v5["id"].str.rsplit("_", n=1, expand=True)
submission_v5["sku"] = id_parts[0]
submission_v5["part"] = id_parts[1]

mask_val = submission_v5["part"] == "validation"
mask_eval = submission_v5["part"] == "evaluation"

for h in range(1, 29):
    validation_date = future_56_dates[h - 1]
    evaluation_date = future_56_dates[h - 1 + 28]

    val_skus = submission_v5.loc[mask_val, "sku"]
    eval_skus = submission_v5.loc[mask_eval, "sku"]

    submission_v5.loc[mask_val, f"F{h}"] = pred_final_56.loc[
        validation_date,
        val_skus
    ].to_numpy(dtype=float)

    submission_v5.loc[mask_eval, f"F{h}"] = pred_final_56.loc[
        evaluation_date,
        eval_skus
    ].to_numpy(dtype=float)

submission_v5 = submission_v5.drop(columns=["sku", "part"])

out_path_v5 = train_path.parent / "submission_v5_direct_lgbm065_current035_56days.csv"
submission_v5.to_csv(out_path_v5, index=False)

print("Saved:", out_path_v5)
print("Shape:", submission_v5.shape)

display(submission_v5.head())
display(submission_v5.tail())

print("Any NaN:", submission_v5.drop(columns=["id"]).isna().any().any())
print("Min prediction:", submission_v5.drop(columns=["id"]).min().min())
print("Max prediction:", submission_v5.drop(columns=["id"]).max().max())

Top SKUs for final direct model: 2000
Origin dates: 2024-03-14 → 2025-08-07
Number of origins: 74
Approx rows: 4144000
X_direct_full: (4144000, 27)
y_direct_full: (4144000,)


TypeError: Invalid value '[[6.03325582e+00 2.93968036e+00 0.00000000e+00 ... 1.25000009e-03
  6.26736763e-02 1.64741542e-01]
 [2.34249973e+00 8.33750069e-01 0.00000000e+00 ... 1.25000009e-03
  7.50000030e-03 6.49999976e-02]
 [1.06585396e+01 5.17861883e+00 1.36312153e-02 ... 1.69635729e-02
  9.19553545e-02 5.40759807e-01]
 ...
 [1.04106701e+01 5.01538969e+00 2.16064021e-02 ... 2.76774800e-02
  3.04985705e-02 1.32689850e-01]
 [1.11634324e+01 4.85719648e+00 1.62198023e-02 ... 3.97533184e-02
  2.99304716e-02 1.52906751e-01]
 [1.02156660e+01 4.02231959e+00 1.62198023e-02 ... 1.93402881e-02
  7.44773258e-02 1.23364563e-01]]' for dtype 'float32'

In [20]:
# =========================
# 19 FIX: TẠO SUBMISSION V5 SAU KHI MODEL ĐÃ TRAIN XONG
# Không cần train lại model_direct_full
# =========================

ALPHA_DIRECT = 0.65
ALPHA_BASE = 0.35

future_56_dates = pd.date_range(
    last_date + pd.Timedelta(days=1),
    periods=56,
    freq="D"
)

future_val_dates = future_56_dates[:28]
future_eval_dates = future_56_dates[28:]

# Base dùng logic current baseline public tốt nhất
pred_base_56 = make_forecast_matrix(
    history=pivot,
    forecast_dates=future_56_dates
)

# FIX QUAN TRỌNG: ép sang float64 để tránh lỗi pandas dtype
pred_final_56 = pred_base_56.astype("float64").copy()

# =========================
# DỰ BÁO DIRECT CHO 28 NGÀY ĐẦU
# =========================

pred_direct_val_top = predict_direct_lgbm(
    model=model_direct_full,
    history_df=pivot,
    origin_date=last_date,
    forecast_dates=future_val_dates,
    sku_list=top_skus_final,
    weights_series=global_weights
)

blend_val = (
    ALPHA_DIRECT * pred_direct_val_top[top_skus_final].astype("float64")
    + ALPHA_BASE * pred_base_56.loc[future_val_dates, top_skus_final].astype("float64")
)

pred_final_56.loc[future_val_dates, top_skus_final] = blend_val.to_numpy(dtype="float64")

# =========================
# DỰ BÁO DIRECT CHO 28 NGÀY TIẾP THEO
# Dùng forecast 28 ngày đầu làm history giả lập
# =========================

augmented_top_history = pivot[top_skus_final].clip(lower=0).astype("float64").copy()

predicted_val_top = pred_final_56.loc[future_val_dates, top_skus_final].copy()

augmented_top_history = pd.concat(
    [augmented_top_history, predicted_val_top],
    axis=0
)

pred_direct_eval_top = predict_direct_lgbm(
    model=model_direct_full,
    history_df=augmented_top_history,
    origin_date=future_val_dates[-1],
    forecast_dates=future_eval_dates,
    sku_list=top_skus_final,
    weights_series=global_weights
)

blend_eval = (
    ALPHA_DIRECT * pred_direct_eval_top[top_skus_final].astype("float64")
    + ALPHA_BASE * pred_base_56.loc[future_eval_dates, top_skus_final].astype("float64")
)

pred_final_56.loc[future_eval_dates, top_skus_final] = blend_eval.to_numpy(dtype="float64")

pred_final_56 = pred_final_56.clip(lower=0)

# =========================
# SET ZERO-WEIGHT SKU = 0
# =========================

zero_weight_skus = weight_table[weight_table["weight"] == 0]["sku"].tolist()
zero_weight_skus = [s for s in zero_weight_skus if s in pred_final_56.columns]

pred_final_56.loc[:, zero_weight_skus] = 0.0

print("Zero-weight SKUs set to 0:", len(zero_weight_skus))

# =========================
# BUILD SUBMISSION FILE
# =========================

submission_v5 = pd.DataFrame()
submission_v5["id"] = sample["id"].values

for h in range(1, 29):
    submission_v5[f"F{h}"] = 0.0

id_parts = submission_v5["id"].str.rsplit("_", n=1, expand=True)
submission_v5["sku"] = id_parts[0]
submission_v5["part"] = id_parts[1]

mask_val = submission_v5["part"] == "validation"
mask_eval = submission_v5["part"] == "evaluation"

for h in range(1, 29):
    validation_date = future_56_dates[h - 1]
    evaluation_date = future_56_dates[h - 1 + 28]

    val_skus = submission_v5.loc[mask_val, "sku"]
    eval_skus = submission_v5.loc[mask_eval, "sku"]

    submission_v5.loc[mask_val, f"F{h}"] = pred_final_56.loc[
        validation_date,
        val_skus
    ].to_numpy(dtype="float64")

    submission_v5.loc[mask_eval, f"F{h}"] = pred_final_56.loc[
        evaluation_date,
        eval_skus
    ].to_numpy(dtype="float64")

submission_v5 = submission_v5.drop(columns=["sku", "part"])

out_path_v5 = train_path.parent / "submission_v5_direct_lgbm065_current035_56days.csv"
submission_v5.to_csv(out_path_v5, index=False)

print("Saved:", out_path_v5)
print("Shape:", submission_v5.shape)

display(submission_v5.head())
display(submission_v5.tail())

print("Any NaN:", submission_v5.drop(columns=["id"]).isna().any().any())
print("Min prediction:", submission_v5.drop(columns=["id"]).min().min())
print("Max prediction:", submission_v5.drop(columns=["id"]).max().max())

Zero-weight SKUs set to 0: 1816
Saved: hbaac-round2\submission_v5_direct_lgbm065_current035_56days.csv
Shape: (31944, 29)


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0.088246,0.04750,0.143476,0.164382,0.261687,0.146433,0.117586,0.113894,0.047500,...,0.160347,0.201985,0.119615,0.115979,0.047741,0.146310,0.167296,0.168364,0.151933,0.119998
1,SKU-00002_validation,2.939680,0.83375,5.178619,4.702252,4.488499,5.044489,4.798717,4.272585,0.849641,...,4.101976,5.096108,3.819647,4.130815,0.881184,5.987980,5.832039,5.015390,4.857197,4.022320
2,SKU-00003_validation,6.033256,2.34250,10.658539,11.448207,11.990372,12.315008,11.722281,7.944589,2.380070,...,11.859054,13.736546,11.896559,10.511451,2.436230,14.487632,14.706127,10.410670,11.163433,10.215666
3,SKU-00004_validation,0.000000,0.00000,0.055275,0.052110,0.053135,0.051674,0.052070,0.036744,0.000000,...,0.075458,0.074785,0.074088,0.042282,0.000000,0.084206,0.091547,0.116220,0.070671,0.069978
4,SKU-00005_validation,0.000000,0.00000,0.017897,0.015314,0.016285,0.015289,0.015665,0.008157,0.000000,...,0.021247,0.021017,0.020372,0.012200,0.000000,0.021716,0.019276,0.028329,0.023180,0.022533


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
31939,SKU-16329_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31940,SKU-16330_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31941,SKU-16331_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31942,SKU-16332_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31943,SKU-16333_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Any NaN: False
Min prediction: 0.0
Max prediction: 156.79581865582713
